<a href="https://colab.research.google.com/github/PULKlT/rag-scrape-educosys.com/blob/main/Simple_rag_scrape_educosys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Install Dependencies & System Requirements
# COMMENT: Consolidated all installations. Removed OpenAI and LangSmith to guarantee a 100% free, local stack.
# COMMENT: Added pciutils so the Ollama installer can actually detect the Colab T4 GPU!
!apt-get update && apt-get install -y zstd pciutils
!pip install -qU langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-chroma langchain-ollama chromadb sentence-transformers beautifulsoup4

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,964 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,889 kB]
Hit:13 https://ppa.launchpadcontent.

In [ ]:
# @title 2. Start Local Ollama Server
import subprocess
import time

# COMMENT: Execute the shell script to install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# COMMENT: Start the server in the background so the notebook cell doesn't hang
subprocess.Popen(["ollama", "serve"])

# COMMENT: Allow the server a brief moment to initialize properly
time.sleep(3)

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# @title 3. Download Llama 3 Model Weights
# COMMENT: Pulling the free local model. This may take a minute depending on network speed.

!ollama pull llama3.2 # using llama3.2 (3B) instead of llama3 (8B) parameters for speed.

In [ ]:
# @title 4A. Install Headless Browser Dependencies
!pip install -qU playwright unstructured nest_asyncio
!playwright install chromium
!playwright install-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 31.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 45.0 MB/s eta 0:00:

In [ ]:
# @title 4B. Load Dynamic Web Documents via Playwright
import nest_asyncio
from langchain_community.document_loaders import PlaywrightURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# COMMENT: Apply nested asyncio to allow Playwright to run cleanly inside Colab
nest_asyncio.apply()

# COMMENT: Point to your target URL
urls = ["https://educosys.com/course-details/genai"]

# COMMENT: Initialize Playwright to evaluate JS and wait for the page to render
loader = PlaywrightURLLoader(
    urls=urls,
    remove_selectors=["header", "footer", "nav"]
)

# COMMENT: Asynchronously load the fully rendered JavaScript page
docs = await loader.aload()

# COMMENT: Standard recursive splitting for optimal retrieval
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(f"Successfully scraped fully-rendered page.")
print(f"Split document into {len(splits)} chunks.")

Successfully scraped fully-rendered page.
Split document into 20 chunks.


In [ ]:
# @title 5. Initialize Free Embeddings & Vector Database
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# COMMENT: Using HuggingFace's free, lightweight, and fast embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}
)
# COMMENT: Utilizing the modern langchain_chroma partner package
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()
print(f"Vector store built successfully with {vectorstore._collection.count()} chunks.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store built successfully with 20 chunks.


In [ ]:
# @title 6. Build the Local LCEL RAG Pipeline
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# COMMENT: Replaced LangSmith hub pull with a standard local prompt to avoid API key requirements
template = """Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Question: {question}

Context: {context}

Answer:"""
prompt = PromptTemplate.from_template(template)

# COMMENT: Initialize the local Ollama instance
llm = ChatOllama(model="llama3.2")

# COMMENT: Helper function to structure retrieved documents
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

# COMMENT: Modern LangChain Expression Language (LCEL) chain structure
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt

    | llm
    | StrOutputParser()
)

print("RAG Pipeline successfully constructed!")

RAG Pipeline successfully constructed!


In [ ]:
# @title 7. Execute Queries
# # COMMENT: Grouped the invoke calls into a clean loop for readable outputs
# questions = [
#     "Are the recordings of the course available for how long?",
#     "Are the testimonials for the course available? Name the students who have shared the testimonials.",
#     "Are there certificates for the courses?",
#     "What all projects are covered in the course?"
# ]

# for q in questions:
#     print(f"Question: {q}")
#     response = rag_chain.invoke(q)
#     print(f"Answer: {response}\n")
#     print("-" * 50 + "\n")

In [ ]:
# @title 7. Execute Queries separately

print("Answer 1:", rag_chain.invoke("Are the recordings of the course available for how long?"))

Answer 1: I don't know for how long the recordings of the course are available. According to the text, it mentions "Lifetime access of Recordings", but no specific duration is provided.


In [ ]:
print("Answer 2:", rag_chain.invoke("Are the testimonials for the course available? Name the students who have shared the testimonials."))

Answer 2: The testimonials are available for the Hands-on Generative AI course, shared by:

1. Raghavi Ramamoorthy
2. Ruthira Sekar (twice)
3. Abhijit Mone
4. Manika Kaushik
5. Gowtamy Reddy Godhala


In [ ]:
print("Answer 3:", rag_chain.invoke("Are there certificates for the courses?"))

Answer 3: I don't know if there are certificates for the courses, but several participants mentioned that they received recognition or confirmation of completion from Educosys upon finishing the course.


In [ ]:
print("Answer 4:", rag_chain.invoke("What all projects are covered in the course?"))

Answer 4: The course covered projects on Generative AI, including hands-on projects that allowed students to apply their knowledge in real-world scenarios. However, the context does not provide specific details on what all projects were covered. It only mentions that 10 hands-on projects were built as part of the course.
